## Sanjula Subhawickrama

# RAG Pipeline — Wikipedia Movie Plots → ChromaDB

Indexes a subset of the [Wikipedia Movie Plots](https://www.kaggle.com/datasets/jrobischon/wikipedia-movie-plots) dataset into a persistent ChromaDB collection using BGE-small embeddings.

This notebook is **ingestion/indexing only** — no LLM calls. Retrieval + generation lives in a separate `02_query.ipynb`.

## Requirements

In [ ]:
%pip install -q pandas chromadb sentence-transformers nltk tqdm

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

## Load & Subset

In [ ]:
import pandas as pd

DATA_PATH = "wiki_movie_plots_deduped.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Raw rows: {len(df_raw)}")

# Keep only the columns we need
KEEP_COLS = ["Title", "Release Year", "Genre", "Director", "Cast", "Plot"]
df = df_raw[KEEP_COLS].copy()

# Drop rows with null/empty Plot or Title
df["Title"] = df["Title"].astype(str).str.strip()
df["Plot"] = df["Plot"].astype(str).str.strip()
df.loc[df["Title"].str.lower().isin(["nan", ""]), "Title"] = None
df.loc[df["Plot"].str.lower().isin(["nan", ""]), "Plot"] = None
df = df.dropna(subset=["Title", "Plot"])
print(f"Rows after dropping null/empty Title or Plot: {len(df)}")

# Sample 300 rows for a manageable subset
df_subset = df.sample(n=300, random_state=42).reset_index(drop=True)
print(f"Subset shape: {df_subset.shape}")
df_subset.head()

## Step 2: Clean & Normalize Metadata

- `Genre`: split on `/` and `,`, lowercase, strip, store as a list (max 3).
- `Cast`: split on `,`, keep the first 3 names as a list.
- `Director`: strip whitespace; `"Unknown"`/empty → `None`.
- `Release Year`: cast to int; drop rows where this fails.
- `Plot`: collapse whitespace/newlines and strip citation artifacts like `[1]`, `[edit]`.

In [ ]:
import re


def clean_genre(genre):
    """Split on '/' and ',', lowercase/strip, keep at most 3 genres."""
    if pd.isna(genre) or str(genre).strip() == "" or str(genre).strip().lower() == "unknown":
        return []
    parts = re.split(r"[/,]", str(genre))
    parts = [p.strip().lower() for p in parts if p.strip()]
    return parts[:3]


def clean_cast(cast):
    """Split on ',', keep the first 3 names."""
    if pd.isna(cast) or str(cast).strip() == "":
        return []
    parts = [p.strip() for p in str(cast).split(",") if p.strip()]
    return parts[:3]


def clean_director(director):
    """Strip whitespace; 'Unknown'/empty -> None."""
    if pd.isna(director):
        return None
    d = str(director).strip()
    if d == "" or d.lower() == "unknown":
        return None
    return d


def clean_plot(plot):
    """Collapse whitespace/newlines and strip citation artifacts like [1], [edit]."""
    text = str(plot)
    text = re.sub(r"\[[^\]]*\]", "", text)  # remove [1], [edit], [citation needed], ...
    text = re.sub(r"\s+", " ", text).strip()
    return text


df_subset["Genre"] = df_subset["Genre"].apply(clean_genre)
df_subset["Cast"] = df_subset["Cast"].apply(clean_cast)
df_subset["Director"] = df_subset["Director"].apply(clean_director)
df_subset["Plot"] = df_subset["Plot"].apply(clean_plot)

# Release Year: cast to int; drop rows where this fails
df_subset["Release Year"] = pd.to_numeric(df_subset["Release Year"], errors="coerce")
before = len(df_subset)
df_subset = df_subset.dropna(subset=["Release Year"]).reset_index(drop=True)
df_subset["Release Year"] = df_subset["Release Year"].astype(int)
print(f"Dropped {before - len(df_subset)} rows with invalid Release Year")

print(f"Cleaned subset shape: {df_subset.shape}")
df_subset.head()